# Ternary similarity map from normalized alignment distance

This notebook is the recommended reference-based view for homologous **protein** sequences. It uses `Biopython`'s `PairwiseAligner` with **BLOSUM62** and affine gap penalties, converts the resulting normalized alignment similarities into a ternary composition, and places each query in the triangle defined by the three references.


## Inputs

This notebook uses the protein FASTA examples bundled in `sequenceTols`:

- `kras_triangle_references.fasta` for the three references,
- `kras_triangle_queries.fasta` for the query sequences.

These examples are derived from homologous K-RAS proteins, so they are compatible with a `BLOSUM62`-based workflow. Replace the paths in the first code cell with your own FASTA files if you want to analyze a different protein family.

This notebook assumes that the input sequences are **proteins**. For nucleotide sequences, this particular workflow is no longer the default recommendation.


In [ ]:
from pathlib import Path
import csv
import math
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
from Bio.Align import PairwiseAligner, substitution_matrices

REFERENCES_FASTA = Path('kras_triangle_references.fasta')
QUERIES_FASTA = Path('kras_triangle_queries.fasta')
OUTPUT_PREFIX = Path('notebook_output')


def parse_fasta(path: Path):
    records = []
    header = None
    chunks = []
    with path.open() as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if header is not None:
                    records.append((header, ''.join(chunks).upper()))
                header = line[1:].strip() or f'seq_{len(records) + 1}'
                chunks = []
            else:
                chunks.append(line)
    if header is not None:
        records.append((header, ''.join(chunks).upper()))
    if not records:
        raise ValueError(f'No FASTA records found in {path}')
    return records


ALIGNER = PairwiseAligner(mode='global')
ALIGNER.substitution_matrix = substitution_matrices.load('BLOSUM62')
ALIGNER.open_gap_score = -10.0
ALIGNER.extend_gap_score = -0.5


def alignment_score(seq_a: str, seq_b: str) -> float:
    return float(ALIGNER.score(seq_a, seq_b))


def normalized_alignment_distance(seq_a: str, seq_b: str):
    score_ab = alignment_score(seq_a, seq_b)
    score_aa = alignment_score(seq_a, seq_a)
    score_bb = alignment_score(seq_b, seq_b)
    scale = math.sqrt(max(score_aa, 1e-12) * max(score_bb, 1e-12))
    similarity = score_ab / scale
    similarity = max(0.0, min(1.0, similarity))
    distance = 1.0 - similarity
    return distance, similarity, score_ab, scale


def close_affinities(affinities):
    affinities = np.asarray(affinities, dtype=float)
    total = float(affinities.sum())
    if total <= 0.0:
        return np.full(3, 1.0 / 3.0)
    return affinities / total


def barycentric_to_cartesian(lambdas):
    vertices = np.array([
        [0.0, 0.0],
        [1.0, 0.0],
        [0.5, math.sqrt(3.0) / 2.0],
    ])
    return np.asarray(lambdas, dtype=float) @ vertices


def write_csv(path: Path, rows, fieldnames):
    with path.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


## Mathematical formulation

Let $S(X,Y)$ be the optimal global pairwise alignment score between protein sequences $X$ and $Y$, computed with **Biopython**'s `PairwiseAligner`, the substitution matrix **BLOSUM62**, and affine gap penalties with gap open $g_o=10$ and gap extend $g_e=0.5$.

For proteins, this is a much better biological dissimilarity model than Hamming or plain Levenshtein distance, because substitutions are weighted according to biochemical plausibility. The normalized alignment distance used here is
\[
d_{\mathrm{align}}(X,Y)=1-rac{S(X,Y)}{\sqrt{S(X,X)\,S(Y,Y)}}.
\]
Equivalently, the normalized alignment similarity is
\[
s_{\mathrm{align}}(X,Y)=rac{S(X,Y)}{\sqrt{S(X,X)\,S(Y,Y)}},
\]
which satisfies $s_{\mathrm{align}}(X,X)=1$ after normalization. In the implementation it is clipped into $[0,1]$ before closure.

For a query $Q$, the three reference similarities are
\[
s_A(Q)=s_{\mathrm{align}}(Q,A),\qquad s_B(Q)=s_{\mathrm{align}}(Q,B),\qquad s_C(Q)=s_{\mathrm{align}}(Q,C).
\]
They are closed into a 3-part composition
\[
\lambda_i(Q)=rac{s_i(Q)}{s_A(Q)+s_B(Q)+s_C(Q)},\qquad i\in\{A,B,C\},
\]
which satisfies $\lambda_A+\lambda_B+\lambda_C=1$. The ternary point is then the barycentric combination
\[
\mathbf{p}(Q)=\lambda_A(Q)V_A+\lambda_B(Q)V_B+\lambda_C(Q)V_C,
\]
with
\[
V_A=(0,0),\qquad V_B=(1,0),\qquad V_C=\left(rac12,rac{\sqrt3}{2}ight).
\]

This is a relative similarity map, not an exact metric embedding of the raw alignment dissimilarities.


## Interpretation and limitations

This notebook preserves the same ternary interpretation as the other triangle notebooks: the plotted point represents **relative similarity to the three references**, not a true metric embedding of sequence space.

Important practical points:

- For homologous proteins, this is the default triangle-based distance in this folder.
- The implementation is intentionally simple and standard: `Biopython PairwiseAligner` + `BLOSUM62` + affine gaps.
- Raw alignment scores are not metrics: triangle inequality can fail, and the geometry depends on the substitution matrix and gap penalties.
- Changing from global to local alignment would change the meaning of the score substantially.
- For very diverged sequences, unstable alignments, recombination, or genome-scale workloads, `k`-mer or model-based distances may be preferable.

So the correct reading is: *how is the biologically weighted alignment similarity of $Q$ distributed relative to $A$, $B$, and $C$?*


In [ ]:
print('Alignment engine: Biopython PairwiseAligner')
print('Substitution matrix: BLOSUM62')
print(f'Gap open: {-ALIGNER.open_gap_score:g}')
print(f'Gap extend: {-ALIGNER.extend_gap_score:g}')


In [ ]:
reference_records = parse_fasta(REFERENCES_FASTA)
query_records = parse_fasta(QUERIES_FASTA)
assert len(reference_records) == 3, 'The references FASTA must contain exactly 3 sequences.'

reference_ids = [record[0] for record in reference_records]
reference_sequences = [record[1] for record in reference_records]
rows = []

for query_id, query_sequence in query_records:
    distances = []
    affinities = []
    raw_scores = []
    scales = []
    for ref_sequence in reference_sequences:
        distance, similarity, score_ab, scale = normalized_alignment_distance(query_sequence, ref_sequence)
        distances.append(distance)
        affinities.append(similarity)
        raw_scores.append(score_ab)
        scales.append(scale)
    distances = np.asarray(distances, dtype=float)
    affinities = np.asarray(affinities, dtype=float)
    lambdas = close_affinities(affinities)
    xy = barycentric_to_cartesian(lambdas)
    rows.append({
        'sequence_id': query_id,
        'distance_to_A': float(distances[0]),
        'distance_to_B': float(distances[1]),
        'distance_to_C': float(distances[2]),
        'similarity_to_A': float(affinities[0]),
        'similarity_to_B': float(affinities[1]),
        'similarity_to_C': float(affinities[2]),
        'raw_score_to_A': float(raw_scores[0]),
        'raw_score_to_B': float(raw_scores[1]),
        'raw_score_to_C': float(raw_scores[2]),
        'scale_to_A': float(scales[0]),
        'scale_to_B': float(scales[1]),
        'scale_to_C': float(scales[2]),
        'lambda_A': float(lambdas[0]),
        'lambda_B': float(lambdas[1]),
        'lambda_C': float(lambdas[2]),
        'x': float(xy[0]),
        'y': float(xy[1]),
    })

fieldnames = list(rows[0].keys())
output_csv = Path(f'{OUTPUT_PREFIX.name}_alignment_substitution_coordinates.csv')
write_csv(output_csv, rows, fieldnames)

vertex_xy = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [0.5, math.sqrt(3.0) / 2.0],
])
query_xy = np.array([[row['x'], row['y']] for row in rows], dtype=float)

fig, ax = plt.subplots(figsize=(7, 7))
triangle = np.vstack((vertex_xy, vertex_xy[0]))
ax.plot(triangle[:, 0], triangle[:, 1], color='black', linewidth=1.5)
ax.scatter(vertex_xy[:, 0], vertex_xy[:, 1], s=180, color='#d04f2a', zorder=3)
ax.scatter(query_xy[:, 0], query_xy[:, 1], s=70, color='#1f6aa5', edgecolor='black', linewidth=0.4, zorder=3)
ax.plot(query_xy[:, 0], query_xy[:, 1], color='#1f6aa5', linewidth=1.0, alpha=0.7, zorder=2)

for label, (x_coord, y_coord) in zip([f'A: {reference_ids[0]}', f'B: {reference_ids[1]}', f'C: {reference_ids[2]}'], vertex_xy):
    ax.text(x_coord, y_coord + 0.04, label, ha='center', va='bottom', fontsize=11, fontweight='bold')

for row in rows:
    ax.text(row['x'] + 0.015, row['y'] + 0.01, row['sequence_id'], fontsize=9)

ax.set_title('Ternary map from normalized Biopython BLOSUM62 alignment distance')
ax.set_xlim(-0.08, 1.08)
ax.set_ylim(-0.08, math.sqrt(3.0) / 2.0 + 0.12)
ax.set_aspect('equal')
ax.axis('off')
fig.tight_layout()

output_png = Path(f'{OUTPUT_PREFIX.name}_alignment_substitution_triangle.png')
fig.savefig(output_png, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved CSV: {output_csv}')
print(f'Saved plot: {output_png}')
pprint(rows)
